# 05 Model: penalised logistic regression vs gradient boosting
Predicts whether a municipality's **next** audit opinion will be **non-reliable** (adverse, disclaimer or outstanding), from the 25 features in `features_muni_year`.

- **Decision 6:** two models, compared. A penalised logistic regression (ridge, lasso or elastic net, whichever validation picks) and gradient boosting (LightGBM).
- **Decision 14:** time-based testing. The models learn from features in 2019–2021 and are tested on 2022–2023, which they never see during training. Settings are tuned with forward-chaining folds inside the training years (learn 2019 and check 2020; learn 2019–2020 and check 2021).
- **Decision 13:** all 25 features go in. The penalty and the trees decide what matters.
- **Main score: PR-AUC** (precision-recall area). It focuses on the rare non-reliable class; random guessing scores about 0.13. Both models are also compared to a **baseline**: "next year will look like this year".

Fabric usually logs scikit-learn runs to its built-in MLflow experiment tracking automatically (autologging).

In [ ]:
import warnings  # lets us hide harmless library notices
warnings.filterwarnings("ignore", category=FutureWarning)  # hide "this option will change in a future version" notices
import numpy as np  # fast maths on lists of numbers
import pandas as pd  # tables in memory; the dataset is small enough to model on one machine
from sklearn.compose import ColumnTransformer  # applies different preparation steps to different columns
from sklearn.pipeline import Pipeline  # chains preparation and model into one object, so test data gets exactly the same treatment
from sklearn.impute import SimpleImputer  # fills gaps (needed by logistic regression)
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # puts numbers on the same scale; turns categories into 0/1 columns
from sklearn.linear_model import LogisticRegression  # the penalised logistic regression
from sklearn.ensemble import HistGradientBoostingClassifier  # fallback gradient boosting if LightGBM isn't installed
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, brier_score_loss  # ways to score predictions
from sklearn.inspection import permutation_importance  # measures how much each feature matters, the same way for both models
try:  # try to use LightGBM, the gradient boosting library we chose...
    from lightgbm import LGBMClassifier  # ...import it if available
    HAVE_LGBM = True  # remember that it worked
except ImportError:  # if it isn't installed...
    HAVE_LGBM = False  # ...we'll use scikit-learn's version, which works the same way

SEED = 42  # fixed random seed, so every run gives the same results
TRAIN_YEARS = [2019, 2020, 2021]  # decision 14: learn from features in these years (predicting opinions 2020-2022)
TEST_YEARS = [2022, 2023]  # decision 14: test on later years the model never saw (predicting opinions 2023-2024)
CATEGORICAL = ["g1_category", "g2_province"]  # text features that need turning into 0/1 columns
NUMERIC = ["a1_opinion_rank", "a2_non_reliable_now", "a3_non_reliable_last3", "a4_years_since_clean", "a5_ever_disclaimer",  # the 23 number features
           "a6_opinion_trend", "b1_missing_income_stmt", "b2_missing_balance_sheet", "b3_missing_debtors", "b4_fin_flagged_broken",
           "c1_irregular_pct", "c2_fruitless_pct", "c3_unauthorised_pct", "c4_irregular_trend",
           "d1_current_ratio", "d2_cash_coverage_months", "d3_operating_margin",
           "e1_debtors_over_90_pct", "e2_debt_impairment_pct", "e3_grant_dependence",
           "f1_employee_cost_pct", "f2_contracted_services_pct", "g3_log_size"]
FEATURES = NUMERIC + CATEGORICAL  # all 25 features (decision 13: no manual selection)

## 1. Load the features and split by year

In [ ]:
df = spark.table("features_muni_year").toPandas()  # copy the features table from the Lakehouse into memory
df = df[df["label"].notna() & df["year"].isin(TRAIN_YEARS + TEST_YEARS)].copy()  # keep labelled rows in the modelling years
df["label"] = df["label"].astype(int)  # the label as 0 (reliable) or 1 (non-reliable)
for c in NUMERIC:  # go through the number features
    df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)  # make sure they're stored as decimal numbers (empty stays empty)
for c in CATEGORICAL:  # go through the text features
    df[c] = df[c].fillna("unknown").astype(str)  # give any gap its own label, so the encoder doesn't trip over it
train = df[df["year"].isin(TRAIN_YEARS)]  # the training rows
test = df[df["year"].isin(TEST_YEARS)]  # the test rows, kept aside until the very end
print(f"train: {len(train)} rows, {train.label.sum()} non-reliable ({train.label.mean():.1%})")  # size and share of non-reliable in training
print(f"test:  {len(test)} rows, {test.label.sum()} non-reliable ({test.label.mean():.1%})")  # same for the test set

# Forward-chaining folds for tuning, inside the training years only: learn 2019, check 2020; then learn 2019-2020, check 2021
folds = [(np.where(train.year <= y - 1)[0], np.where(train.year == y)[0]) for y in TRAIN_YEARS[1:]]  # (rows to learn from, rows to check on) for each fold

## 2. Build both models and tune their settings on the training years only

In [ ]:
def logistic(C, l1_ratio):
    """Build the logistic regression pipeline: fill gaps, add 'was missing' flags, scale, then a penalised model."""
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)),  # fill gaps with the training median and add a 0/1 "was missing" column (keeps decision 5's signal)
                          ("scale", StandardScaler())]), NUMERIC),  # put every number feature on the same scale, so the penalty treats them fairly
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])  # one 0/1 column per category and province
    model = LogisticRegression(penalty="elasticnet", solver="saga", C=C, l1_ratio=l1_ratio,  # elastic net: l1_ratio 0 = ridge, 1 = lasso, in between = a blend
                               class_weight="balanced", max_iter=5000, random_state=SEED)  # "balanced" gives the rare non-reliable class more weight
    return Pipeline([("prep", prep), ("model", model)])  # preparation and model chained together

def boosting(learning_rate, max_depth):
    """Build the gradient boosting pipeline. It handles gaps itself, so only categories need converting."""
    prep = ColumnTransformer([("num", "passthrough", NUMERIC),  # number features go in as they are, gaps included
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL)])  # categories become 0/1 columns
    if HAVE_LGBM:  # LightGBM if installed
        model = LGBMClassifier(n_estimators=300, learning_rate=learning_rate, max_depth=max_depth, num_leaves=15,  # 300 small trees, each correcting the last
                               min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,  # limits that stop it memorising the training data
                               class_weight="balanced", random_state=SEED, verbose=-1)  # extra weight on the rare class; no log spam
    else:  # otherwise scikit-learn's equivalent
        model = HistGradientBoostingClassifier(max_iter=300, learning_rate=learning_rate, max_depth=max_depth,
                                               class_weight="balanced", random_state=SEED)
    return Pipeline([("prep", prep), ("model", model)])  # preparation and model chained together

GRIDS = {  # the settings to try for each model; the forward-chaining folds pick the best
    "logistic": (logistic, [dict(C=C, l1_ratio=r) for C in [0.01, 0.1, 1.0] for r in [0.0, 0.5, 1.0]]),  # strength of the penalty x ridge / elastic net / lasso
    "boosting": (boosting, [dict(learning_rate=lr, max_depth=d) for lr in [0.03, 0.1] for d in [2, 3, 4]]),  # learning speed x tree depth
}

def tune(build, grid):
    """Try every setting on the forward-chaining folds and return the one with the best average PR-AUC."""
    X, y = train[FEATURES], train["label"].to_numpy()  # training features and labels
    results = []  # one score per setting
    for params in grid:  # go through each setting
        scores = []  # one score per fold
        for fit_rows, check_rows in folds:  # each fold: learn on earlier years, check on the next year
            m = build(**params).fit(X.iloc[fit_rows], y[fit_rows])  # train on the earlier years
            p = m.predict_proba(X.iloc[check_rows])[:, 1]  # predicted chance of non-reliable for the next year
            scores.append(average_precision_score(y[check_rows], p))  # PR-AUC: how well it ranks the rare non-reliable cases
        results.append((np.mean(scores), params))  # average over the folds
    best_score, best_params = max(results, key=lambda r: r[0])  # the setting with the highest average
    print(f"  best {best_params}: validation PR-AUC {best_score:.3f}")  # report it
    return best_params

best = {}  # the chosen setting for each model
for name, (build, grid) in GRIDS.items():  # tune each model
    print(name)  # which model
    best[name] = tune(build, grid)  # try the settings and keep the best

## 3. Test on 2022–2023 and compare with the baseline

In [ ]:
def threshold_for_recall(y, p, target=0.8):
    """The highest cut-off that still catches at least `target` of the non-reliable cases."""
    for t in np.sort(np.unique(p))[::-1]:  # try cut-offs from high to low
        if recall_score(y, p >= t) >= target:  # stop at the first that catches enough
            return t
    return 0.0

def scores(y, p, t):
    """All the scores for one set of predictions at cut-off t."""
    flag = p >= t  # which municipalities get flagged
    return {"pr_auc": average_precision_score(y, p),  # ranking quality on the rare class (main measure; 0.135 = random guessing)
            "roc_auc": roc_auc_score(y, p),  # overall ranking quality (0.5 = random)
            "recall": recall_score(y, flag),  # share of truly non-reliable municipalities we caught
            "precision": precision_score(y, flag, zero_division=0),  # share of flagged municipalities that really were non-reliable
            "flagged": int(flag.sum()),  # how many were flagged
            "brier": brier_score_loss(y, p)}  # how accurate the probabilities are (lower is better)

X_tr, y_tr = train[FEATURES], train["label"].to_numpy()  # all training data
X_te, y_te = test[FEATURES], test["label"].to_numpy()  # the untouched test data
fitted, rows, preds = {}, [], test[["demarcation_code", "name", "year", "opinion_code", "label"]].copy()  # trained models, score rows, predictions table

# Baseline: "next year will look like this year". Any model must beat this to be worth it.
base_te = (test["a2_non_reliable_now"].fillna(0) > 0.5).astype(float).to_numpy()  # baseline "probability" in the test years: 1 if non-reliable now, else 0
rows.append({"model": "baseline: same as this year", **scores(y_te, base_te, 0.5)})  # score it on the test years

for name, (build, _) in GRIDS.items():  # the two models
    m = build(**best[name]).fit(X_tr, y_tr)  # retrain with the best setting on all training years
    fitted[name] = m  # keep it
    t = threshold_for_recall(y_tr, m.predict_proba(X_tr)[:, 1])  # choose the cut-off on training data only (aim: catch 80%)
    p = m.predict_proba(X_te)[:, 1]  # predict the test years
    preds[f"prob_{name}"] = p  # store the probabilities
    preds[f"flag_{name}"] = (p >= t).astype(int)  # and whether each municipality is flagged
    rows.append({"model": name, "threshold": t, **scores(y_te, p, t)})  # score on the test years

metrics = pd.DataFrame(rows)  # one row per model
display(metrics.round(3))  # show the comparison

## 4. Explain: what drives each model?
Permutation importance shuffles one feature at a time on the test years and measures how much PR-AUC drops. It's the same yardstick for both models. The logistic weights show direction: + raises the risk, − lowers it.

In [ ]:
imp = []  # one row per model and feature
for name, m in fitted.items():  # for each trained model
    r = permutation_importance(m, X_te, y_te, scoring="average_precision", n_repeats=20, random_state=SEED)  # shuffle one feature at a time and see how much PR-AUC drops
    for f, mean, sd in zip(FEATURES, r.importances_mean, r.importances_std):  # each feature's average drop and its spread
        imp.append({"model": name, "feature": f, "importance": mean, "importance_sd": sd})  # store it
imp = pd.DataFrame(imp)  # as a table
display(imp.pivot(index="feature", columns="model", values="importance").sort_values("boosting", ascending=False).round(4))  # side by side, biggest first

lr = fitted["logistic"]  # the logistic model
names = lr.named_steps["prep"].get_feature_names_out()  # the names of its inputs after preparation (including "was missing" flags)
coef = pd.Series(lr.named_steps["model"].coef_[0], index=names)  # one weight per input: + raises the risk, - lowers it (per standard deviation)
print("logistic: inputs kept (non-zero weight):", int((coef != 0).sum()), "of", len(coef))  # lasso can set weights to exactly zero
display(coef[coef != 0].sort_values(key=abs, ascending=False).head(15).round(3).to_frame("weight"))  # the 15 biggest weights

## 5. Save results to the Lakehouse (for Power BI)

In [ ]:
spark.createDataFrame(preds).write.mode("overwrite").format("delta").saveAsTable("model_predictions")  # test-year predictions, for Power BI
spark.createDataFrame(metrics.fillna(0)).write.mode("overwrite").format("delta").saveAsTable("model_metrics")  # the score comparison
spark.createDataFrame(imp).write.mode("overwrite").format("delta").saveAsTable("model_feature_importance")  # what drives each model
print("saved model_predictions, model_metrics, model_feature_importance")  # confirm

## 6. Checks

In [ ]:
def check(name, ok, detail=""):
    """Print PASS or FAIL for one check, with optional detail in brackets."""
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))

check("train and test years don't overlap", set(train.year) & set(test.year) == set(), sorted(set(test.year)))  # decision 14: no shared years
check("test set is the 2022-2023 rows", len(test) == (df.year.isin(TEST_YEARS)).sum(), len(test))  # nothing dropped
check("every test row has both predictions", preds[["prob_logistic", "prob_boosting"]].notna().all().all())  # no gaps in the output
check("probabilities between 0 and 1", preds[["prob_logistic", "prob_boosting"]].stack().between(0, 1).all())  # sane probabilities
check("25 features used", len(FEATURES) == 25, len(FEATURES))  # decision 13
check("baseline and both models scored", list(metrics["model"]) == ["baseline: same as this year", "logistic", "boosting"], len(metrics))  # three rows in the comparison